# LV6 - klasifikacija - knn i svm

### K Nearest Neighbours

- knn / k najbližih susjeda
- može se koristiti i za regresiju i za klasifikaciju - mi ćemo ga koristiti primarno za klasifikaciju
- neparametarski algoritam = ne traži aproksimirajuću funkciju između X i y, nego predviđa neki primjerak izravno na temelju skupa za učenje
- dobar je je za razliku od logističke regresije jer stvara nelinearnu granicu odluke -> može dobro klasificirati podatke koji nisu linearno odvojivi
- radi tako da promatra K najbližih susjeda svih primjeraka i ovisno o njihovoj klasi, klasificira i te primjerke
- udaljenost se može računati koristeći različite metrike, npr. euklidska ili manhattan udaljenost
- metriku i k zovemo hiperparametrima jer njih određujemo pri instanciranju klase modela
- bitno je dobro odabrati hiperparametar k - prevelika vrijednost dovest će do podnaučenja (underfitting), a premali k prouzrokovat će prenaučenje (overfitting)
- s obzirom da knn klasificira primjerke tako što računa udaljenost od okolnih primjeraka, vrlo je važno skalirati podatke

In [2]:
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor

### Support Vector Machine
- svm / stroj s potpornim vektorima
- također se može koristiti i za klasifikaciju i regresiju
- temelji se na maksimizaciji granice odluke ("margine") između podataka za učenje koji su blizu granice odluke
- ti podaci za učenje nazivaju se potporni vektori
- umjesto svih podatkovnih primjeraka, potrebni su mu samo oni koji su blizu granice odluke -> puno efikasniji od knn
- s obzirom da podaci često nisu linearno odvojivi, svm ima hiperparametar c koji će "moderirati" količinu pogrešno klasificiranih primjeraka
- s obzirom da je svm-u u cilju pronaći što širu marginu, time će vjerojatno dio primjeraka pogrešno klasificirati, što će hiperparametar c korigirati
- veći c = veća penalizacija pogrešne klasifikacije
- osim c, svm ima i hiperparametar kernel - mi smo najčešće koristili rbf, poly i linear
- rbf kernel dodatno ima hiperparametar gama, a poly ima hiperparametar degree

In [3]:
from sklearn.svm import SVC, SVR

### Odabir optimalnog modela

- s obzirom da ovaj veliki broj mogućih hiperparametara, javlja se problem odabira modela koji neće niti prenaučiti niti podnaučiti podatke

#### Validacija
- optimalni model možemo odabrati korištenjem skupa podataka za validaciju
- to je manji skup koji se izdvaja iz skupa za učenje i služi za procjenu performansi modela
- za to ne smijemo koristiti skup za testiranje jer on služi za finalnu procjenu performansi
- s obzirom da smo već dio podataka izdvojili za testiranje i ostalo nam je 70-80% podataka unutar skupa za učenje, skup za validaciju ne smije biti prevelik jer onda nećemo imati dovoljno podataka za učenje modela
- međutim, možemo obaviti unakrsnu validaciju - podijeliti skup na učenje na k dijelova i samo jedan koristiti za validaciju, te provesti taj postupak k puta

In [4]:
from sklearn.model_selection import cross_val_score

#### Grid Search

- Grid Search klasa nam omogućava da navedemo različite željene hiperparametre za model i njih odjednom isprobamo
- bitno nam je izabrati dobre hiperparametre jer ovisno o njima model može biti prejednostavan pa ne naučiti dobro podatke, ili pretjerano kompleksan pa prenaučiti podatke i imati loše rezultate na testnom skupu
- klasa će nam dati najbolju kombinaciju hiperparametara i rezultat koji su ostvarili na validacijskom skupu
- to nam je korisno jer metodom .fit() možemo obaviti unakrsnu validaciju za sve kombinacije navedenih hiperparametara i ne moramo ju raditi sami
- prima parametre estimator (instanca modela kojeg želimo koristiti) i cv (broj podskupova skupa za učenje)

In [5]:
from sklearn.model_selection import GridSearchCV

Sve zajedno:

In [6]:
import pandas as pd

data = pd.read_csv('Social_Network_Ads.csv')

data = data.drop(columns='User ID') # nije nam bitan za učenje
data = pd.get_dummies(data, columns=['Gender'])

X = data.drop(columns='Purchased')
y = data['Purchased']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [7]:
svm = KNeighborsClassifier()
svm_grid = {
    'n_neighbors' : [1, 3, 5, 10, 20],
    'metric' : ['minkowski', 'manhattan', 'euclidean']
}

best_svm = GridSearchCV(estimator=svm, param_grid=svm_grid, cv=5, scoring='accuracy')
best_svm.fit(X_train, y_train)

print('Najbolji hiperparametri: ', best_svm.best_params_)
print('Najbolji rezultat: ', best_svm.best_score_)

Najbolji hiperparametri:  {'metric': 'minkowski', 'n_neighbors': 3}
Najbolji rezultat:  0.9125


In [8]:
svm = SVC()
svm_grid = {
    'C' : [0.01, 0.1, 1],
    'kernel' : ['poly', 'rbf', 'linear'],
    'degree' : [1, 2, 3, 5],
    'gamma' : [0.01, 0.1, 1]
}

best_svm = GridSearchCV(estimator=svm, param_grid=svm_grid, cv=5, scoring='accuracy')
best_svm.fit(X_train, y_train)

print('Najbolji hiperparametri: ', best_svm.best_params_)
print('Najbolji rezultat: ', best_svm.best_score_)

Najbolji hiperparametri:  {'C': 1, 'degree': 1, 'gamma': 0.1, 'kernel': 'rbf'}
Najbolji rezultat:  0.915625
